# Week 3: Model Training and Hyperparameter Tuning
In this notebook, we continue from the data preprocessing stage completed in Week 2. Our goal now is to train a machine learning model to predict cardiovascular disease, and optimize its performance using hyperparameter tuning.

## Step 1: Import Necessary Libraries
We need libraries for data manipulation, visualization, and machine learning components from `sklearn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For splitting data, building the model, tuning, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

## Step 2: Load the Preprocessed Data
We load the `cardio_train_cleaned.csv` dataset, which contains scaled and outlier-free data prepared during Week 2.

In [ ]:
# Load the cleaned dataset
df = pd.read_csv("cardio_train_cleaned.csv")

# Display the first 5 rows to ensure it loaded correctly
df.head()

## Step 3: Train-Test Split
**Purpose:** Machine learning models must be tested on unseen data to ensure they don't just memorize the training dataset (a problem known as overfitting).

- We separate our features (`X`) from our target variable (`y`, which is the `cardio` column).
- We split the dataset so 80% is used for training the model, and 20% is held back for testing.

In [ ]:
# Separate features and target
X = df.drop("cardio", axis=1) # All columns except 'cardio' are features
y = df["cardio"]            # 'cardio' is what we want to predict

# Split the data: 80% for training, 20% for testing. 
# random_state ensures reproducibility (we get the same split every time we run this)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data features shape:", X_train.shape)
print("Testing data features shape:", X_test.shape)

## Step 4: Base Model Training (Random Forest)
**Purpose:** Before spending time optimizing a model, it's good practice to train a "baseline" model using default settings. This gives us a benchmark to see if our tuning actually improves performance.

We use **Random Forest** because it is robust, handles complex relationships well, and is less prone to overfitting than a single decision tree.

In [ ]:
# Initialize the Random Forest model with default parameters
rf_base = RandomForestClassifier(random_state=42)

# Train (fit) the model on the training data
rf_base.fit(X_train, y_train)

# Predict the target variable for our testing data
y_pred_base = rf_base.predict(X_test)

# Evaluate the baseline model's performance
print("Base Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("\nClassification Report:\n", classification_report(y_test, y_pred_base))

## Step 5: Hyperparameter Tuning with RandomizedSearchCV
**Purpose:** Default model settings are rarely the absolute best. Hyperparameters are the "knobs and dials" of the algorithm.

- `n_estimators`: Number of trees in the forest.
- `max_depth`: How deep each tree can grow. Limiting this prevents overfitting.
- `min_samples_split` & `min_samples_leaf`: Limits on how nodes are split.

Instead of trying every possible combination (which takes a very long time), we use `RandomizedSearchCV` to randomly sample combinations and find a highly optimal configuration much faster.

In [ ]:
# Define the grid of hyperparameters we want to search through
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# Set up Randomized Search
# Changed n_jobs to 1 to avoid "No space left on device" memmapping issues
rf_random = RandomizedSearchCV(estimator=rf_base, param_distributions=param_dist, 
                               n_iter=20, cv=3, verbose=1, random_state=42, n_jobs=1)

# Start the search (this might take a minute or two!)
rf_random.fit(X_train, y_train)

# Display the best "knobs and dials" found during the search
print("Best Parameters Found:", rf_random.best_params_)

## Step 6: Evaluate the Tuned Model
**Purpose:** Now we take the "best" model found by the RandomizedSearch and see how it performs on our hold-out test set. We compare this to our baseline model from Step 4.

In [ ]:
# Extract the best estimator (model) from our search
best_rf = rf_random.best_estimator_

# Predict on the test set using the tuned model
y_pred_tuned = best_rf.predict(X_test)

# Output the improved metrics
print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("\nTuned Classification Report:\n", classification_report(y_test, y_pred_tuned))

## Step 7: Confusion Matrix
**Purpose:** Accuracy isn't everything. A confusion matrix shows us exactly *where* our model is making mistakes. 
- **True Positives/Negatives:** Correct predictions.
- **False Positives:** Model predicted disease, but patient is healthy.
- **False Negatives:** Model predicted healthy, but patient has disease.

In [ ]:
# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)

# Plot it using a heatmap for easy interpretation
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Tuned Model)")
plt.xlabel("Predicted Label (0 = No Disease, 1 = Disease)")
plt.ylabel("True Label (0 = No Disease, 1 = Disease)")
plt.show()

## Step 8: ROC Curve Analysis
Let's analyze the train vs test AUC to check for overfitting/underfitting.

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get the predicted probabilities for the positive class (disease = 1)
y_train_probs = best_rf.predict_proba(X_train)[:, 1]
y_test_probs = best_rf.predict_proba(X_test)[:, 1]

# Calculate False Positive Rate (fpr) and True Positive Rate (tpr) for both sets
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_probs)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_probs)

# Calculate Area Under the Curve (AUC)
auc_train = auc(fpr_train, tpr_train)
auc_test = auc(fpr_test, tpr_test)

# Output for evaluation
print(f"Train AUC: {auc_train:.3f}")
print(f"Test AUC: {auc_test:.3f}")

# Plot the ROC curves
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, color='blue', label=f'Train ROC curve (AUC = {auc_train:.3f})')
plt.plot(fpr_test, tpr_test, color='red', label=f'Test ROC curve (AUC = {auc_test:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') # Diagonal line (random guessing)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

## Step 9: Cross-Validation Evaluation
To ensure our model's performance is consistent and not dependent on a specific train/test split, we evaluate the tuned model using 5-fold cross-validation across the entire dataset.

In [ ]:
from sklearn.model_selection import cross_val_score

# Evaluate the best model using 5-fold cross-validation on the entire dataset
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy')

print(f"Cross-Validation Accuracy Scores (5 folds): {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation of CV Accuracy: {cv_scores.std():.4f}")